# №1

In [34]:
import cv2
import numpy as np

In [35]:
K = np.array([[1200, 0, 640],
              [0, 1200, 360],
              [0, 0, 1]])

np.random.seed(42)
n_points = 100

world_points = np.random.rand(n_points, 3) * 10

R_true = cv2.Rodrigues(np.array([0.2, -0.1, 0.05]))[0]
t_true = np.array([[0.5], [-0.2], [0.1]])

pts1_3d, _ = cv2.projectPoints(world_points, np.zeros((3, 1)), np.zeros((3, 1)), K, None)
pts1 = pts1_3d.reshape(-1, 2).astype(np.float32)

pts2_3d, _ = cv2.projectPoints(world_points, R_true, t_true, K, None)
pts2 = pts2_3d.reshape(-1, 2).astype(np.float32)

# додаємо шум
pts1 += np.random.randn(n_points, 2) * 1.5
pts2 += np.random.randn(n_points, 2) * 1.5

e_matrix, mask = cv2.findEssentialMat(pts1, pts2, K)
print("E: ", e_matrix, sep="\n")

_, R, t, mask = cv2.recoverPose(e_matrix, pts1, pts2, K)
print("R: ", R, "t: ", t, sep="\n")

E: 
[[-0.03393542 -0.17650346 -0.24269067]
 [ 0.05806333 -0.13250878 -0.63532364]
 [ 0.29976988  0.60884312 -0.15387715]]
R: 
[[ 0.99365302 -0.06187253 -0.09394394]
 [ 0.04201107  0.97883057 -0.20031419]
 [ 0.10434915  0.19509612  0.97521729]]
t: 
[[ 0.90420811]
 [-0.38841716]
 [ 0.17759448]]


In [36]:
_, rvec, tvec = cv2.solvePnP(world_points, pts2, K, None)

R_pnp, _ = cv2.Rodrigues(rvec)

print("R (PnP):", R_pnp, "t (PnP):", tvec, sep="\n")

print("\n--- Порівняння ---")
print("Справжній R:", cv2.Rodrigues(rvec_true)[0])
print("Справжній t:", tvec_true.T)

R (PnP):
[[ 0.99378231 -0.05948084 -0.09412089]
 [ 0.03958169  0.97885793 -0.20067495]
 [ 0.10406729  0.19570176  0.97512605]]
t (PnP):
[[ 0.49984466]
 [-0.20048554]
 [ 0.10059092]]

--- Порівняння ---
Справжній R: [[ 0.9937773  -0.05951997 -0.09414913]
 [ 0.03960732  0.97884281 -0.20074367]
 [ 0.10410546  0.19576551  0.97510918]]
Справжній t: [[ 0.5 -0.2  0.1]]


# №2

In [37]:
def gram_schmidt(R):
    u1 = R[:, 0]
    u2 = R[:, 1]
    u3 = R[:, 2]
    
    e1 = u1 / np.linalg.norm(u1)
    
    u2 = u2 - np.dot(u2, e1) * e1
    e2 = u2 / np.linalg.norm(u2)
    
    u3 = u3 - np.dot(u3, e1) * e1 - np.dot(u3, e2) * e2
    e3 = u3 / np.linalg.norm(u3)
    
    R_ortho = np.column_stack((e1, e2, e3))
    
    if np.linalg.det(R_ortho) < 0:
        e3 = -e3
        R_ortho = np.column_stack((e1, e2, e3))
    
    return R_ortho

R_refined = gram_schmidt(R)
print("R refined:\n", R_refined)

R refined:
 [[ 0.99365302 -0.06187253 -0.09394394]
 [ 0.04201107  0.97883057 -0.20031419]
 [ 0.10434915  0.19509612  0.97521729]]
